In [ ]:
# ==========================================
# 1. MOUNT GOOGLE DRIVE & SETUP
# ==========================================
from google.colab import drive
import os
import ee
import pandas as pd
import numpy as np

drive.mount('/content/drive')

try:
    ee.Initialize(project='integrated-hawk-485001-k3')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='integrated-hawk-485001-k3')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# 2. DEFINE GEOMETRIES & BUFFER LOGIC
# ==========================================
ASSET_ID   = 'projects/integrated-hawk-485001-k3/assets/PH_DHS_GPS'
dhs_points = ee.FeatureCollection(ASSET_ID)

print(f"Total DHS clusters: {dhs_points.size().getInfo()}")

def adaptive_buffer(feature):
    """
    Buffers each DHS point by 2km (urban) or 5km (rural).
    CRITICAL FIX: use copyProperties() to preserve DHSCLUST
    and other properties on the buffered feature.
    """
    urban_rural = ee.String(feature.get('URBAN_RURA'))
    is_urban    = urban_rural.compareTo('U').eq(0)
    radius      = ee.Number(ee.Algorithms.If(is_urban, 2000, 5000))
    buffered    = feature.buffer(radius).bounds()

    # Preserve all original properties on the new geometry
    return ee.Feature(buffered.geometry(),
                      feature.toDictionary())

dhs_squares = dhs_points.map(adaptive_buffer)

# Quick sanity check that DHSCLUST survived the buffer
sample = dhs_squares.first().getInfo()
print(f"Sample buffered feature properties: "
      f"{list(sample['properties'].keys())}")
assert 'DHSCLUST' in sample['properties'], \
    "DHSCLUST missing from buffered features. Check adaptive_buffer."


Total DHS clusters: 1247
Sample buffered feature properties: ['ADM1DHS', 'ADM1FIPS', 'ADM1FIPSNA', 'ADM1NAME', 'ADM1SALBCO', 'ADM1SALBNA', 'ALT_DEM', 'ALT_GPS', 'CCFIPS', 'DATUM', 'DHSCC', 'DHSCLUST', 'DHSID', 'DHSREGCO', 'DHSREGNA', 'DHSYEAR', 'LATNUM', 'LONGNUM', 'SOURCE', 'URBAN_RURA']


In [ ]:
# ==========================================
# 3. VIIRS PROCESSING (correct collection)
# ==========================================
# Using VNP46A1 daily DNB product with stray-light correction.
# This matches your manuscript methodology and your original
# phase1-ntl-median.ipynb approach.
# Band: DNB_At_Sensor_Radiance_500m (nW/cm²/sr)

print("Building VIIRS 2022 annual median composite...")

viirs_2022_median = (
    ee.ImageCollection('NOAA/VIIRS/001/VNP46A1')
    .filterDate('2022-01-01', '2022-12-31')
    .select('DNB_At_Sensor_Radiance_500m')
    .median()
)

# Verify the composite has data
sample_val = viirs_2022_median.reduceRegion(
    reducer  = ee.Reducer.mean(),
    geometry = dhs_squares.first().geometry(),
    scale    = 500
).getInfo()
print(f"Sample VIIRS value at first cluster: {sample_val}")


Building VIIRS 2022 annual median composite...
Sample VIIRS value at first cluster: {'DNB_At_Sensor_Radiance_500m': 4.066734284496124}


In [ ]:
# ==========================================
# 4. REDUCE REGIONS AND EXPORT TO DRIVE
# ==========================================
# Use Export instead of getInfo() to avoid timeout on 1247 clusters.
# This exports a CSV directly to your Drive.

OUTPUT_DIR   = '/content/drive/MyDrive/Thesis_Data'
OUTPUT_CSV   = os.path.join(OUTPUT_DIR, 'viirs_ntl_labels_all_clusters.csv')
GCS_BUCKET   = 'gs://tala-sentinel2-data'
EXPORT_TABLE = 'viirs_ntl_labels_export'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Submitting GEE export task...")
print("This computes on GEE servers and saves to Drive.")
print("Estimated time: 5-15 minutes depending on GEE queue.")

viirs_reduced = viirs_2022_median.reduceRegions(
    collection = dhs_squares,
    reducer    = ee.Reducer.median(),
    scale      = 500       # Native VIIRS resolution
)

# Keep only the columns we need to reduce file size
viirs_export = viirs_reduced.select(
    propertySelectors = ['DHSCLUST', 'URBAN_RURA', 'median'],
    retainGeometry    = False
)

# Export to Drive as CSV
export_task = ee.batch.Export.table.toDrive(
    collection      = viirs_export,
    description     = EXPORT_TABLE,
    folder          = 'Thesis_Data',
    fileNamePrefix  = 'viirs_ntl_labels_raw',
    fileFormat      = 'CSV'
)
export_task.start()
print(f"Export task started. Task ID: {export_task.id}")
print("Monitor at: https://code.earthengine.google.com/tasks")


Submitting GEE export task...
This computes on GEE servers and saves to Drive.
Estimated time: 5-15 minutes depending on GEE queue.
Export task started. Task ID: 6GQVELLUJHYQQ7TSQEZZARNV
Monitor at: https://code.earthengine.google.com/tasks


In [ ]:
# ==========================================
# 5. POLL UNTIL EXPORT COMPLETES
# ==========================================
import time

print("\nWaiting for export to complete...")
print("(Checks every 30 seconds. Safe to interrupt and check manually.)")

while True:
    status = export_task.status()
    state  = status['state']
    print(f"  [{time.strftime('%H:%M:%S')}] Status: {state}")

    if state == 'COMPLETED':
        print("Export complete.")
        break
    elif state in ('FAILED', 'CANCELLED'):
        print(f"Export failed: {status.get('error_message', 'unknown error')}")
        raise RuntimeError(f"GEE export failed: {state}")
    else:
        time.sleep(30)


Waiting for export to complete...
(Checks every 30 seconds. Safe to interrupt and check manually.)
  [05:24:17] Status: READY
  [05:24:47] Status: READY
  [05:25:17] Status: READY
  [05:25:48] Status: READY
  [05:26:18] Status: READY
  [05:26:48] Status: READY
  [05:27:18] Status: READY
  [05:27:49] Status: READY
  [05:28:19] Status: READY
  [05:28:50] Status: READY
  [05:29:20] Status: READY
  [05:29:51] Status: READY
  [05:30:21] Status: READY
  [05:30:51] Status: RUNNING
  [05:31:21] Status: RUNNING
  [05:31:52] Status: RUNNING
  [05:32:22] Status: RUNNING
  [05:32:52] Status: RUNNING
  [05:33:23] Status: RUNNING
  [05:33:53] Status: COMPLETED
Export complete.


In [16]:
# ==========================================
# 6. LOAD EXPORTED CSV AND COMPUTE CLASSES
# ==========================================
# The exported CSV will be in Drive/Thesis_Data/
# The column name from reduceRegions is 'median'

raw_csv = os.path.join(OUTPUT_DIR, 'viirs_ntl_labels_raw.csv')

# Wait briefly for Drive to sync
time.sleep(5)

if not os.path.exists(raw_csv):
    raise FileNotFoundError(
        f"CSV not found at {raw_csv}. "
        "Check Drive/Thesis_Data/ for the file. "
        "It may take a minute for Drive to sync."
    )

df_raw = pd.read_csv(raw_csv)
print(f"\nLoaded {len(df_raw)} rows from exported CSV")
print(f"Columns: {list(df_raw.columns)}")
print(f"Sample:\n{df_raw.head()}")


Loaded 1247 rows from exported CSV
Columns: ['system:index', 'DHSCLUST', 'URBAN_RURA', 'median', '.geo']
Sample:
           system:index  DHSCLUST URBAN_RURA    median  \
0  0000000000000000022e     559.0          U  3.830396   
1  0000000000000000042a    1067.0          R  0.700000   
2  00000000000000000443    1092.0          R  0.545927   
3  00000000000000000462    1123.0          R  0.900000   
4  00000000000000000442    1091.0          U  1.039681   

                                     .geo  
0  {"type":"MultiPoint","coordinates":[]}  
1  {"type":"MultiPoint","coordinates":[]}  
2  {"type":"MultiPoint","coordinates":[]}  
3  {"type":"MultiPoint","coordinates":[]}  
4  {"type":"MultiPoint","coordinates":[]}  


In [17]:
# ==========================================
# 7. CLEAN AND VALIDATE
# ==========================================
# Rename median column to NTL_Value for clarity
df = df_raw[['DHSCLUST', 'URBAN_RURA', 'median']].copy()
df = df.rename(columns={'median': 'NTL_Value'})
df['DHSCLUST'] = df['DHSCLUST'].astype(int)

# Check for missing values
n_missing = df['NTL_Value'].isna().sum()
print(f"\nMissing NTL values: {n_missing} clusters")

if n_missing > 0:
    print("Clusters with missing VIIRS data:")
    print(df[df['NTL_Value'].isna()][['DHSCLUST', 'URBAN_RURA']])
    print("\nStrategy: fill missing with 0 (treat as dark).")
    print("These are likely very remote clusters with persistent")
    print("cloud cover. Marking as Dark (class 0) is conservative.")
    df['NTL_Value'] = df['NTL_Value'].fillna(0)

# Basic stats
print(f"\nNTL Value statistics:")
print(df['NTL_Value'].describe().round(4))
print(f"Zeros (possibly dark or missing): "
      f"{(df['NTL_Value'] == 0).sum()}")



Missing NTL values: 0 clusters

NTL Value statistics:
count    1247.0000
mean        4.0233
std         7.6134
min         0.5371
25%         0.5672
50%         0.7000
75%         2.6571
max        42.3049
Name: NTL_Value, dtype: float64
Zeros (possibly dark or missing): 0


In [18]:
# ==========================================
# 8. COMPUTE TERCILE NTL CLASSES
# ==========================================
# pd.qcut fails if too many duplicate values at boundaries.
# Use try/except with duplicates='drop' as fallback.

print("\nComputing NTL tercile classes...")

try:
    df['NTL_Class'] = pd.qcut(
        df['NTL_Value'],
        q      = 3,
        labels = [0, 1, 2]
    )
    print("Tercile boundaries (qcut):")
    print(pd.qcut(df['NTL_Value'], q=3).value_counts().sort_index())

except ValueError as e:
    print(f"qcut failed ({e}), using duplicates='drop' fallback")
    df['NTL_Class'] = pd.qcut(
        df['NTL_Value'],
        q          = 3,
        labels     = [0, 1, 2],
        duplicates = 'drop'
    )

df['NTL_Class'] = df['NTL_Class'].astype(int)

# Verify class distribution
print(f"\nClass distribution:")
dist = df['NTL_Class'].value_counts().sort_index()
for cls, name in [(0,'Dark'), (1,'Dim'), (2,'Bright')]:
    count = dist.get(cls, 0)
    pct   = count / len(df) * 100
    print(f"  Class {cls} ({name:6s}): {count:4d} ({pct:.1f}%)")

# Warn if any class has fewer than 10% of samples
for cls in [0, 1, 2]:
    pct = dist.get(cls, 0) / len(df) * 100
    if pct < 10:
        print(f"WARNING: Class {cls} has only {pct:.1f}% of samples.")
        print("Consider adjusting class boundaries or using "
              "manual thresholds instead of qcut.")



Computing NTL tercile classes...
Tercile boundaries (qcut):
NTL_Value
(0.536, 0.576]     416
(0.576, 1.351]     415
(1.351, 42.305]    416
Name: count, dtype: int64

Class distribution:
  Class 0 (Dark  ):  416 (33.4%)
  Class 1 (Dim   ):  415 (33.3%)
  Class 2 (Bright):  416 (33.4%)


In [19]:
# ==========================================
# 9. SAVE FINAL OUTPUT
# ==========================================
df.to_csv(OUTPUT_CSV, index=False)

print("\n" + "="*50)
print("PROCESSING SUMMARY")
print("="*50)
print(f"Total clusters processed : {len(df)}")
print(f"NTL value range          : "
      f"[{df['NTL_Value'].min():.4f}, "
      f"{df['NTL_Value'].max():.4f}] nW/cm²/sr")
print(f"Class distribution       :")
print(df['NTL_Class'].value_counts().sort_index().to_string())
print(f"\nSaved to: {OUTPUT_CSV}")
print("\nNext step: use this CSV as input to")
print("phase1-proxy-cnn-tpu.ipynb (labels_file path)")


PROCESSING SUMMARY
Total clusters processed : 1247
NTL value range          : [0.5371, 42.3049] nW/cm²/sr
Class distribution       :
NTL_Class
0    416
1    415
2    416

Saved to: /content/drive/MyDrive/Thesis_Data/viirs_ntl_labels_all_clusters.csv

Next step: use this CSV as input to
phase1-proxy-cnn-tpu.ipynb (labels_file path)
